<a href="https://colab.research.google.com/github/VitorBZS/PLN-A940-M-D.S.M.-297-20262/blob/main/Aula_Pratica_Passo_a_Passo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# >>>>> TROQUE AQUI O MOTOR: "qwen" (na aula, com GPU) ou "gemini" (em casa, com chave) <<<<<
MOTOR = "qwen"

!pip install -q spacy scikit-learn gradio transformers accelerate google-genai
!python -m spacy download pt_core_news_sm -q

import re
import unicodedata
import pandas as pd
import spacy
import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("pt_core_news_sm")
print("Ambiente configurado. Motor escolhido:", MOTOR)


✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Ambiente configurado. Motor escolhido: qwen


In [2]:
# ===== Bot da Parte 1 (base, regras, intenção, entidades, busca e memória) =====

imoveis = pd.DataFrame([

    ["AP01", "apartamento", "Aluguel", "Matão",        "Centro",                     2,   1300, False],

    ["AP02", "apartamento", "Aluguel", "Matão",        "Jardim Paraíso",             3,   1800, False],

    ["AP03", "apartamento", "Venda",   "Matão",        "Centro",                     2, 280000, True],

    ["CA01", "casa",        "Aluguel", "Matão",        "Vila Santa Cruz",            2,   1100, False],

    ["CA02", "casa",        "Venda",   "Matão",        "Residencial Olívio Benassi", 3, 420000, True],

    ["CA03", "casa",        "Venda",   "Matão",        "Jardim Bela Vista",          2, 250000, True],

    ["KT01", "kitnet",      "Aluguel", "Matão",        "Centro",                     1,    750, False],

    ["AP04", "apartamento", "Aluguel", "Araraquara",   "Centro",                     2,   1600, False],

    ["AP05", "apartamento", "Venda",   "Araraquara",   "Fonte",                      3, 390000, True],

    ["CA04", "casa",        "Aluguel", "Araraquara",   "Vila Xavier",                3,   2200, False],

    ["CA05", "casa",        "Venda",   "Araraquara",   "Jardim Primavera",           2, 310000, True],

    ["CA06", "casa",        "Venda",   "Taquaritinga", "Centro",                     3, 350000, True],

    ["CA07", "casa",        "Aluguel", "Taquaritinga", "Jardim Buscardi",            2,   1000, False],

    ["AP06", "apartamento", "Venda",   "Taquaritinga", "Centro",                     2, 230000, True],

    ["TE01", "terreno",     "Venda",   "Matão",        "Jardim Paraíso",             0, 120000, True],

], columns=["codigo", "tipo", "finalidade", "cidade", "bairro", "quartos", "valor", "aceita_financiamento"])

def normalizar(texto):

    texto = unicodedata.normalize("NFD", texto.lower())

    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

REGRAS = {

    "saudacao":   (["oi", "ola", "bom dia", "boa tarde", "boa noite"],

                   "Olá! Sou o assistente virtual da imobiliária. Posso ajudar você a comprar ou alugar um imóvel."),

    "horario":    (["horario", "abre", "funciona", "expediente"],

                   "Atendemos de segunda a sexta, das 8h às 18h, e aos sábados, das 8h às 12h."),

    "fgts":       (["fgts"],

                   "Sim! O FGTS pode ser usado na compra do imóvel, desde que o comprador atenda às regras da Caixa."),

    "documentos": (["documento", "documentos", "documentacao"],

                   "Para alugar: RG, CPF, comprovante de renda e de residência. Para comprar, o corretor envia a lista completa."),

    "despedida":  (["obrigado", "obrigada", "tchau", "valeu"],

                   "Eu que agradeço! Quando quiser, é só chamar."),

}

def responder_regras(mensagem):

    texto = normalizar(mensagem)   # normalização (Seção 2)

    palavras = set(re.findall(r"\w+", texto))   # tokenização com expressão regular

    for nome, (gatilhos, resposta) in REGRAS.items():

        for gatilho in gatilhos:

            if (" " in gatilho and gatilho in texto) or gatilho in palavras:

                return resposta

    return None

imoveis_ex = ["o apartamento", "a casa", "o sobrado", "o studio", "o terreno", "a cobertura"]

modelos_compra = ["Quero comprar {}, aceita financiamento?", "Qual o valor de venda d{}?",

                  "Tenho interesse em adquirir {} à vista.", "Posso usar o FGTS para comprar {}?",

                  "Qual a entrada mínima para financiar {}?"]

modelos_aluguel = ["{} ainda está disponível para alugar?", "Qual o valor do aluguel d{}?",

                   "Aceitam fiador para alugar {}?", "{} para locação aceita pet?",

                   "Qual o prazo do contrato de aluguel d{}?"]

def montar(modelo, imovel):

    if "d{}" in modelo:

        return modelo.replace("d{}", "d" + imovel)

    frase = modelo.format(imovel)

    return frase[0].upper() + frase[1:]

textos_intencao = [montar(m, i) for m in modelos_compra for i in imoveis_ex] + [montar(m, i) for m in modelos_aluguel for i in imoveis_ex]

rotulos_intencao = ["Compra"] * 30 + ["Aluguel"] * 30

PALAVRAS_NEGACAO = {"não", "nunca", "jamais", "nada", "ninguém"}

def lematizar(texto):

    doc = nlp(texto)

    return " ".join(

        token.lemma_.lower() for token in doc

        if not token.is_punct and (not token.is_stop or token.lower_ in PALAVRAS_NEGACAO)

    )

vetorizador = TfidfVectorizer()   # TF-IDF (aula de Features)

X_intencao = vetorizador.fit_transform([lematizar(t) for t in textos_intencao])   # lematização + vetorização

modelo_intencao = LogisticRegression(max_iter=1000)   # Regressão Logística (ML Parte 1)

modelo_intencao.fit(X_intencao, rotulos_intencao)   # treino (ML Parte 1)

LIMIAR = 0.6

def classificar_intencao(mensagem):

    vetor = vetorizador.transform([lematizar(mensagem)])

    probabilidades = modelo_intencao.predict_proba(vetor)[0]

    posicao = probabilidades.argmax()

    return modelo_intencao.classes_[posicao], probabilidades[posicao]

TIPOS = {"apartamento": "apartamento", "apto": "apartamento", "casa": "casa",

         "kitnet": "kitnet", "terreno": "terreno", "lote": "terreno"}

CIDADES = ["Matão", "Araraquara", "Taquaritinga"]

def extrair_entidades(mensagem):

    texto = normalizar(mensagem)   # normalização (Seção 2)

    palavras = set(re.findall(r"\w+", texto))

    entidades = {"tipo": None, "cidade": None, "quartos": None, "valor_max": None}

    for palavra, tipo in TIPOS.items():   # tipo: lista de palavras

        if palavra in palavras:

            entidades["tipo"] = tipo

    for cidade in CIDADES:   # cidade: lista (no lugar do NER)

        if normalizar(cidade) in texto:

            entidades["cidade"] = cidade

    achou = re.search(r"(\d+)\s*quartos?", texto)   # quartos: expressão regular

    if achou:

        entidades["quartos"] = int(achou.group(1))

    achou = re.search(r"ate\s*(?:r\$\s*)?(\d+(?:\.\d{3})*)\s*(mil)?", texto)   # valor: expressão regular

    if achou:

        valor = int(achou.group(1).replace(".", ""))

        if achou.group(2):

            valor = valor * 1000

        entidades["valor_max"] = valor

    return entidades

def buscar_imoveis(finalidade, entidades):

    resultado = imoveis

    if finalidade:

        resultado = resultado[resultado["finalidade"] == finalidade]

    if entidades["tipo"]:

        resultado = resultado[resultado["tipo"] == entidades["tipo"]]

    if entidades["cidade"]:

        resultado = resultado[resultado["cidade"] == entidades["cidade"]]

    if entidades["quartos"]:

        resultado = resultado[resultado["quartos"] >= entidades["quartos"]]

    if entidades["valor_max"]:

        resultado = resultado[resultado["valor"] <= entidades["valor_max"]]

    return resultado

def formatar_valor(valor, finalidade):

    texto = f"R$ {valor:,.0f}".replace(",", ".")

    return texto + "/mês" if finalidade == "Aluguel" else texto

def montar_resposta(resultado):

    if len(resultado) == 0:

        return "Não encontrei imóveis com essas características. Quer tentar outra cidade ou outro valor?"

    linhas = [f"Encontrei {len(resultado)} imóvel(is):"]

    for _, imovel in resultado.head(3).iterrows():

        linhas.append(f"• {imovel['codigo']}: {imovel['tipo']}, {imovel['quartos']} quarto(s), "

                      f"{imovel['bairro']}, {imovel['cidade']}: {formatar_valor(imovel['valor'], imovel['finalidade'])}")

    if len(resultado) > 3:

        linhas.append(f"...e mais {len(resultado) - 3}. Quer que um corretor entre em contato?")

    return "\n".join(linhas)

def responder_pedido(finalidade, entidades):

    resultado = buscar_imoveis(finalidade, entidades)

    if len(resultado) == 0 and entidades["tipo"]:

        sem_tipo = entidades.copy()

        sem_tipo["tipo"] = None

        alternativas = buscar_imoveis(finalidade, sem_tipo)

        if len(alternativas) > 0:

            return (f"Não encontrei {entidades['tipo']} com essas características, "

                    "mas tenho outras opções.\n" + montar_resposta(alternativas))

    return montar_resposta(resultado)

FINALIDADE = {"Compra": "Venda", "Aluguel": "Aluguel"}

def juntar_pedido(mensagens):

    finalidade = None

    entidades = {"tipo": None, "cidade": None, "quartos": None, "valor_max": None}

    for m in mensagens:

        detalhes = extrair_entidades(m)   # Camada 3: entidades (Seção 4)

        if responder_regras(m) and not any(detalhes.values()):   # Camada 1: regras (Seção 2)

            continue

        intencao, confianca = classificar_intencao(m)   # Camada 2: intenção (Seção 3)

        if confianca >= LIMIAR:   # limiar de confiança (Seção 3.1)

            finalidade = FINALIDADE[intencao]

        for chave, valor in detalhes.items():

            if valor:

                entidades[chave] = valor

    return finalidade, entidades

def chatbot(mensagem, anteriores=None):

    if anteriores is None:

        anteriores = []

    resposta_regra = responder_regras(mensagem)

    tem_detalhes = any(extrair_entidades(mensagem).values())

    intencao, confianca = classificar_intencao(mensagem)

    if resposta_regra and not tem_detalhes:

        return resposta_regra

    if not tem_detalhes and confianca < LIMIAR:

        return "Desculpe, não entendi. Você procura um imóvel para comprar ou para alugar?"

    finalidade, entidades = juntar_pedido(anteriores + [mensagem])   # memória: todas as mensagens do lead

    if finalidade is None:

        return "Entendi o que você procura! É para comprar ou para alugar?"

    resposta = responder_pedido(finalidade, entidades)   # Camada 4: busca (Seção 5)

    if resposta_regra:

        resposta = resposta_regra + "\n" + resposta

    return resposta

print("Bot da Parte 1 pronto:", len(imoveis), "imóveis na base.")


Bot da Parte 1 pronto: 15 imóveis na base.


In [3]:
conversas_dificeis = [
    ["Quero alugar um apartamento em Matão", "Pode ser qualquer tipo, não precisa ser apartamento"],
    ["Para alugar o que tem em Taquaritinga?", "Tem algum imóvel até 150 mil?"],
    ["Vocês atendem que horas?"],
    ["Tem alguma coisa barata pra morar sozinho?"],
]

def testar_conversa(funcao, mensagens):
    anteriores = []
    for m in mensagens:
        print("Lead:", m)
        print("Bot: ", funcao(m, anteriores).replace("\n", "\n      "))
        print()
        anteriores.append(m)

for conversa in conversas_dificeis:
    testar_conversa(chatbot, conversa)
    print("-" * 70)

Lead: Quero alugar um apartamento em Matão
Bot:  Encontrei 2 imóvel(is):
      • AP01: apartamento, 2 quarto(s), Centro, Matão: R$ 1.300/mês
      • AP02: apartamento, 3 quarto(s), Jardim Paraíso, Matão: R$ 1.800/mês

Lead: Pode ser qualquer tipo, não precisa ser apartamento
Bot:  Encontrei 2 imóvel(is):
      • AP01: apartamento, 2 quarto(s), Centro, Matão: R$ 1.300/mês
      • AP02: apartamento, 3 quarto(s), Jardim Paraíso, Matão: R$ 1.800/mês

----------------------------------------------------------------------
Lead: Para alugar o que tem em Taquaritinga?
Bot:  Encontrei 1 imóvel(is):
      • CA07: casa, 2 quarto(s), Jardim Buscardi, Taquaritinga: R$ 1.000/mês

Lead: Tem algum imóvel até 150 mil?
Bot:  Encontrei 1 imóvel(is):
      • CA07: casa, 2 quarto(s), Jardim Buscardi, Taquaritinga: R$ 1.000/mês

----------------------------------------------------------------------
Lead: Vocês atendem que horas?
Bot:  Desculpe, não entendi. Você procura um imóvel para comprar ou para alugar

In [4]:
if MOTOR == "qwen":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    NOME_QWEN = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(NOME_QWEN)
    modelo = AutoModelForCausalLM.from_pretrained(NOME_QWEN, device_map="auto")

    def gerar(sistema, conversa):
        mensagens = [{"role": "system", "content": sistema}] + conversa
        texto = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
        entrada = tokenizer([texto], return_tensors="pt").to(modelo.device)
        with torch.no_grad():
            saida = modelo.generate(**entrada, max_new_tokens=250, do_sample=False)
        novos = saida[0][entrada["input_ids"].shape[1]:]
        return tokenizer.decode(novos, skip_special_tokens=True).strip()

elif MOTOR == "gemini":
    import time
    from google import genai
    from google.genai import types
    from google.colab import userdata

    # se um modelo estiver sobrecarregado ou indisponível, o notebook tenta o seguinte
    MODELOS_GEMINI = ["gemini-flash-latest", "gemini-flash-lite-latest", "gemini-3.5-flash"]
    cliente = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

    def gerar(sistema, conversa):
        # a conversa vai como um único texto: "Lead: ..." / "Assistente: ..."
        nomes = {"user": "Lead", "assistant": "Assistente"}
        texto = "\n".join(f"{nomes[m['role']]}: {m['content']}" for m in conversa)
        configuracao = types.GenerateContentConfig(
            system_instruction=sistema,
            http_options=types.HttpOptions(timeout=60000),   # no máximo 60 s por tentativa
        )
        for modelo_gemini in MODELOS_GEMINI:
            for tentativa in range(2):
                try:
                    resposta = cliente.models.generate_content(
                        model=modelo_gemini,
                        contents=texto + "\nAssistente:",
                        config=configuracao,
                    )
                    return resposta.text.strip()
                except Exception as erro:
                    if ("timeout" in type(erro).__name__.lower() or "timed out" in str(erro).lower()
                            or "504" in str(erro) or "DEADLINE_EXCEEDED" in str(erro)):
                        print(f"[{modelo_gemini} demorou demais; passando para o próximo]")
                        break
                    if "404" in str(erro) or "NOT_FOUND" in str(erro).upper():
                        print(f"[{modelo_gemini} indisponível; passando para o próximo]")
                        break
                    temporario = any(x in str(erro) for x in ("500", "502", "503", "429", "INTERNAL", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "high demand", "overloaded"))
                    if not temporario:
                        raise
                    print(f"[{modelo_gemini} ocupado; tentando de novo...]")
                    time.sleep(3)
        return "Desculpe, o serviço está sobrecarregado agora. Tente novamente em alguns minutos."

print("Motor", MOTOR, "carregado.")
print(gerar("Responda em português, em uma frase.", [{"role": "user", "content": "O que é uma imobiliária?"}]))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Motor qwen carregado.
Uma imobiliária é um tipo de empresa especializada na comercialização e venda de imóveis, incluindo casas, apartamentos, terrenos e outros tipos de propriedades.


In [6]:
SISTEMA_SEM_BASE = "Você é o assistente virtual de uma imobiliária de Matão (SP). Responda em português, de forma curta e cordial."

pergunta = "Quais apartamentos vocês têm para alugar em Matão e quanto custam?"

print("Lead:", pergunta)

print("Bot: ", gerar(SISTEMA_SEM_BASE, [{"role": "user", "content": pergunta}]))


Lead: Quais apartamentos vocês têm para alugar em Matão e quanto custam?
Bot:  Olá! Infelizmente não tenho informações específicas sobre os preços ou disponibilidade atual dos apartamentos para aluguel em Matão. Recomendo que você visite diretamente a imobiliária local para obter essas informações mais precisas. Boa sorte na sua busca!


In [7]:
FAQ = [

    "Horário de atendimento (que horas a imobiliária atende): de segunda a sexta, das 8h às 18h, e aos sábados, das 8h às 12h.",

    "Documentos para alugar: RG, CPF, comprovante de renda e comprovante de residência.",

    "Garantias aceitas no aluguel: fiador com imóvel na cidade, seguro-fiança ou caução de três aluguéis.",

    "FGTS: pode ser usado na compra do imóvel residencial, desde que o comprador atenda às regras da Caixa.",

    "Financiamento: os imóveis à venda marcados como aceita financiamento podem ser financiados; a entrada mínima costuma ser de 20%.",

    "Visitas: são agendadas com um corretor, de segunda a sábado, com pelo menos um dia de antecedência.",

    "Animais de estimação (cachorro, gato): a maioria dos imóveis para alugar aceita animais; o corretor confirma para cada imóvel.",

    "Condomínio e IPTU: no aluguel, são pagos pelo inquilino, além do valor do aluguel.",

]

vetorizador_faq = TfidfVectorizer()

matriz_faq = vetorizador_faq.fit_transform([lematizar(f) for f in FAQ])

def buscar_faq(pergunta, quantidade=2):

    vetor = vetorizador_faq.transform([lematizar(pergunta)])

    notas = cosine_similarity(vetor, matriz_faq)[0]

    melhores = notas.argsort()[::-1][:quantidade]

    return [FAQ[i] for i in melhores if notas[i] > 0]

for p in ["Precisa de fiador para alugar?", "Vocês atendem que horas?", "Posso ter cachorro no apartamento?"]:

    print(p, "->", buscar_faq(p))


Precisa de fiador para alugar? -> ['Garantias aceitas no aluguel: fiador com imóvel na cidade, seguro-fiança ou caução de três aluguéis.', 'Documentos para alugar: RG, CPF, comprovante de renda e comprovante de residência.']
Vocês atendem que horas? -> ['Horário de atendimento (que horas a imobiliária atende): de segunda a sexta, das 8h às 18h, e aos sábados, das 8h às 12h.']
Posso ter cachorro no apartamento? -> ['Animais de estimação (cachorro, gato): a maioria dos imóveis para alugar aceita animais; o corretor confirma para cada imóvel.']


In [8]:
def imovel_em_texto(imovel):
    preco = formatar_valor(imovel["valor"], imovel["finalidade"])
    quartos = "sem quartos" if imovel["quartos"] == 0 else f"{imovel['quartos']} quarto(s)"
    financia = ", aceita financiamento" if imovel["aceita_financiamento"] else ""
    return (f"{imovel['codigo']}: {imovel['tipo']} para {imovel['finalidade'].lower()}, "
            f"{quartos}, bairro {imovel['bairro']}, {imovel['cidade']}, {preco}{financia}")

BASE_EM_TEXTO = "\n".join(imovel_em_texto(imovel) for _, imovel in imoveis.iterrows())
print(BASE_EM_TEXTO)

AP01: apartamento para aluguel, 2 quarto(s), bairro Centro, Matão, R$ 1.300/mês
AP02: apartamento para aluguel, 3 quarto(s), bairro Jardim Paraíso, Matão, R$ 1.800/mês
AP03: apartamento para venda, 2 quarto(s), bairro Centro, Matão, R$ 280.000, aceita financiamento
CA01: casa para aluguel, 2 quarto(s), bairro Vila Santa Cruz, Matão, R$ 1.100/mês
CA02: casa para venda, 3 quarto(s), bairro Residencial Olívio Benassi, Matão, R$ 420.000, aceita financiamento
CA03: casa para venda, 2 quarto(s), bairro Jardim Bela Vista, Matão, R$ 250.000, aceita financiamento
KT01: kitnet para aluguel, 1 quarto(s), bairro Centro, Matão, R$ 750/mês
AP04: apartamento para aluguel, 2 quarto(s), bairro Centro, Araraquara, R$ 1.600/mês
AP05: apartamento para venda, 3 quarto(s), bairro Fonte, Araraquara, R$ 390.000, aceita financiamento
CA04: casa para aluguel, 3 quarto(s), bairro Vila Xavier, Araraquara, R$ 2.200/mês
CA05: casa para venda, 2 quarto(s), bairro Jardim Primavera, Araraquara, R$ 310.000, aceita fina

In [9]:
INSTRUCOES = """Você é o assistente virtual de uma imobiliária que atende Matão, Araraquara e Taquaritinga (SP).
Regras:
- Responda sempre em português, de forma curta e cordial.
- Use SOMENTE as informações do CONTEXTO abaixo. Nunca invente imóveis, valores, bairros ou condições.
- Ao citar um imóvel, informe o código, o tipo, a cidade e o valor.
- Se a informação não estiver no contexto, diga que vai confirmar com um corretor.
- Se não souber se o lead quer comprar ou alugar, pergunte."""

def montar_sistema(pergunta):
    perguntas = buscar_faq(pergunta)
    contexto = "IMÓVEIS DISPONÍVEIS:\n" + BASE_EM_TEXTO
    if perguntas:
        contexto += "\n\nINFORMAÇÕES DA IMOBILIÁRIA:\n" + "\n".join(perguntas)
    return INSTRUCOES + "\n\nCONTEXTO:\n" + contexto

def chatbot_generativo(mensagem, conversa_anterior=None):
    if conversa_anterior is None:
        conversa_anterior = []
    conversa = conversa_anterior + [{"role": "user", "content": mensagem}]
    return gerar(montar_sistema(mensagem), conversa)

pergunta = "Quais apartamentos vocês têm para alugar em Matão e quanto custam?"
print("Lead:", pergunta)
print("Bot: ", chatbot_generativo(pergunta))

Lead: Quais apartamentos vocês têm para alugar em Matão e quanto custam?
Bot:  Vamos ver os apartamentos disponíveis para alugar em Matão:

1. AP01 - Apartamento para aluguel, 2 quartos, bairro Centro, Matão, R$ 1.300/mês

Estes são os apartamentos disponíveis para alugar em Matão. O valor mensal é de R$ 1.300 por mês.


In [10]:
def testar_conversa_generativa(mensagens):
    conversa = []
    for m in mensagens:
        resposta = chatbot_generativo(m, conversa)
        print("Lead:", m)
        print("Bot: ", resposta.replace("\n", "\n      "))
        print()
        conversa += [{"role": "user", "content": m}, {"role": "assistant", "content": resposta}]

for conversa in conversas_dificeis:
    testar_conversa_generativa(conversa)
    print("-" * 70)

Lead: Quero alugar um apartamento em Matão
Bot:  O apartamento para aluguel está localizado no bairro Centro e custa R$ 1.300 por mês.

Lead: Pode ser qualquer tipo, não precisa ser apartamento
Bot:  Claro! O apartamento para aluguel está localizado no bairro Centro e custa R$ 1.300 por mês.

----------------------------------------------------------------------
Lead: Para alugar o que tem em Taquaritinga?
Bot:  Tem dois imóveis disponíveis para alugar em Taquaritinga:
      
      - Casa para aluguel: 2 quartos, bairro Jardim Bela Vista, Taquaritinga, R$ 250.000, aceita financiamento.
      - Casa para aluguel: 2 quartos, bairro Jardim Buscardi, Taquaritinga, R$ 1.000/mês.

Lead: Tem algum imóvel até 150 mil?
Bot:  Sim, tem um imóvel disponível para alugar até R$ 150.000 em Taquaritinga:
      
      Casa para aluguel: 2 quartos, bairro Jardim Bela Vista, Taquaritinga, R$ 250.000, aceita financiamento.

----------------------------------------------------------------------
Lead: Vocês

In [11]:
def texto_da_mensagem(conteudo):

    if isinstance(conteudo, list):

        conteudo = " ".join(parte["text"] for parte in conteudo

                            if isinstance(parte, dict) and "text" in parte)

    return conteudo if isinstance(conteudo, str) else ""

def historico_em_conversa(historico):

    conversa = []

    for item in historico:

        if isinstance(item, dict):

            texto = texto_da_mensagem(item["content"])

            if texto and item["role"] in ("user", "assistant"):

                conversa.append({"role": item["role"], "content": texto})

        else:

            conversa += [{"role": "user", "content": item[0]}, {"role": "assistant", "content": item[1]}]

    return conversa

def responder(mensagem, historico):

    return chatbot_generativo(mensagem, historico_em_conversa(historico))

tela = gr.ChatInterface(

    fn=responder,

    title="Assistente Virtual — Imobiliária (com IA generativa)",

    description="Pergunte sobre imóveis para comprar ou alugar em Matão, Araraquara e Taquaritinga.",

    examples=["Oi, bom dia!", "Quero alugar um apartamento em Matão",

              "Pode ser qualquer tipo, não precisa ser apartamento"],

)

tela.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e59075d77237bfa286.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://e59075d77237bfa286.gradio.live
